In [1]:

#  1. Install Required Packages
!pip install mujoco imageio pillow stable-baselines3[extra] shimmy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 2.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 2.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [shimmy]2m1/3 [ale-py]


In [ ]:
#  2. Moun Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving 22.tar.gz to 22.tar.gz


In [ ]:
!tar -xzf /content/22.tar.gz

In [ ]:

# ✅ 4. Configure Headless EGL Rendering
import os
os.environ["MUJOCO_GL"] = "egl"

In [5]:
#  5. Import Libraries
import mujoco
from mujoco import MjModel, MjData
import numpy as np
import imageio
from PIL import Image
from stable_baselines3 import TD3
from stable_baselines3.common.callbacks import BaseCallback
import gymnasium as gym
from gymnasium import spaces, Env


In [6]:
#  6. Load Your Robot XML Model
model = MjModel.from_xml_path("/home/oussema/Documents/project/results/g1_combined_ultra_stable.xml")
data = MjData(model)

In [ ]:
!ls /content/allegro_hand_pick_package/assets/

 allegro_hand_pick.xml
 assets
 cube1.xml
 cube_copy.xml
 cube.xml
'g1_dual_arm _copy.xml'
 g1_dual_arm_fixed_hands.xml
 g1_dual_arm_friction_ready.xml
 g1_dual_arm_patched.xml
 g1_dual_arm_powered_hands.xml
 g1_dual_arm_RIGHT_ONLY_PATCHED.xml
 g1_dual_arm_RIGHT_ONLY.xml
 g1_dual_arm_simplified_FIXED_PATCHED.xml
 g1_dual_arm_simplified_fixed.xml
 g1_dual_arm_simplified_PATCHED_GRIP.xml
 g1_dual_arm_simplified.xml
 g1_dual_arm_with_cube.xml
 g1_dual_arm.xml
 g1_with_hands_torso_only_static.xml
 g1_with_hands_torso_only.xml
 g1_with_hands.xml
 left_hand-old.xml
 left_hand.xml
 meshes
 right_hand.xml
 scene_left.xml


In [7]:
#  Optional: Lighting / Visual Tweaks
model.vis.quality.shadowsize = 4096
model.vis.global_.offwidth = 640
model.vis.global_.offheight = 480

In [8]:
for i in range(model.nbody):
    print(i, model.body(i).name)

0 world
1 table
2 cube
3 left_shoulder_pitch_link
4 left_shoulder_roll_link
5 left_shoulder_yaw_link
6 left_elbow_link
7 left_wrist_roll_link
8 left_wrist_pitch_link
9 left_wrist_yaw_link
10 right_shoulder_pitch_link
11 right_shoulder_roll_link
12 right_shoulder_yaw_link
13 right_elbow_link
14 right_wrist_roll_link
15 right_wrist_pitch_link
16 right_wrist_yaw_link
17 left_index_0
18 left_index_1
19 left_middle_0
20 left_middle_1
21 left_ring_0
22 left_ring_1
23 left_thumb_0
24 left_thumb_1
25 right_index_0
26 right_index_1
27 right_middle_0
28 right_middle_1
29 right_ring_0
30 right_ring_1
31 right_thumb_0
32 right_thumb_1
33 left_thumb_force_sensors
34 left_index_force_sensors
35 left_middle_force_sensors
36 left_ring_force_sensors
37 right_thumb_force_sensors
38 right_index_force_sensors
39 right_middle_force_sensors
40 right_ring_force_sensors


In [37]:
class CustomRobotEnv(Env):
    def __init__(self, render_mode=None, eval_mode=False):
        super().__init__()
        self.render_mode = render_mode
        self.eval_mode = eval_mode
        self.model = model
        self.data = data

        # Actuators: right side only
        right_actuators = []
        for i in range(self.model.nu):
            name = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
            if name is not None and name.startswith("right_"):
                right_actuators.append(i)
        self.right_actuator_ids = np.array(right_actuators, dtype=np.int32)

        self.action_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(len(self.right_actuator_ids),),
            dtype=np.float32
        )

        self.renderer = mujoco.Renderer(self.model, width=640, height=480)

        obs_dim = self.model.nq + self.model.nv + 9
        self.observation_space = spaces.Box(
            low=-1e10, high=1e10,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.current_step = 0
        self.max_steps = 500
        self.success_counter = 0
        self.freeze_timer = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        mujoco.mj_forward(self.model, self.data)

        self.current_step = 0

        cube_joint_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, "cube:joint")
        cube_qpos_addr = self.model.jnt_qposadr[cube_joint_id]

        fixed_cube_pos = np.array([0.18, 0.0, 0.04])
        start = cube_qpos_addr
        end = cube_qpos_addr + 3
        if end <= len(self.data.qpos):
            self.data.qpos[start:end] = fixed_cube_pos
        else:
            size = len(self.data.qpos) - start
            if size > 0:
                self.data.qpos[start:start + size] = fixed_cube_pos[:size]

        fixed_cube_quat = np.array([1, 0, 0, 0])
        start = cube_qpos_addr + 3
        end = cube_qpos_addr + 7
        if end <= len(self.data.qpos):
            self.data.qpos[start:end] = fixed_cube_quat
        else:
            size = len(self.data.qpos) - start
            if size > 0:
                self.data.qpos[start:start + size] = fixed_cube_quat[:size]

        obs = self._get_obs()
        return obs, {}





    def step(self, action):
      # Split action
      arm_action = action[:7]
      finger_action = action[7:]

      # Get positions
      cube_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")
      cube_pos = self.data.xpos[cube_id]
      palm_pos = self.data.body("right_hand_index_1_link").xpos
      thumb_pos = self.data.body("right_hand_thumb_2_link").xpos
      index_pos = self.data.body("right_hand_index_1_link").xpos
      middle_pos = self.data.body("right_hand_middle_1_link").xpos

      # Distances
      dist = np.linalg.norm(palm_pos - cube_pos)
      thumb_dist = np.linalg.norm(thumb_pos - cube_pos)
      index_dist = np.linalg.norm(index_pos - cube_pos)
      middle_dist = np.linalg.norm(middle_pos - cube_pos)

      # Contact detection
      thumb_contact = self._is_touching("cube_geom", "right_hand_thumb_2_geom")
      index_contact = self._is_touching("cube_geom", "right_hand_index_1_geom")
      middle_contact = self._is_touching("cube_geom", "right_hand_middle_1_geom")
      num_contacts = sum([thumb_contact, index_contact, middle_contact])

      # Scale actions based on distance
      ARM_SCALE = 0.4 if dist > 0.08 else 0.2
      FINGER_SCALE = 0.7

      # Reset controls
      self.data.ctrl[:] = 0.0

      # Apply scaled actions
      self.data.ctrl[self.right_actuator_ids[:7]] = arm_action * ARM_SCALE
      self.data.ctrl[self.right_actuator_ids[7:]] = finger_action * FINGER_SCALE

      # Grasp assist: encourage closure when 2 fingers are near
      if dist < 0.06 and num_contacts >= 2:
          assist_strength = 0.5
          self.data.ctrl[self.right_actuator_ids[7:]] += assist_strength
          self.data.ctrl[self.right_actuator_ids[7:]] = np.clip(
              self.data.ctrl[self.right_actuator_ids[7:]], -1.0, 1.0
         )
          print("🤝 Grasp assist triggered (≥2 fingers touching)")

      # Step simulation
      mujoco.mj_step(self.model, self.data)
      obs = self._get_obs()
      reward = self._compute_reward()
      self.current_step += 1

      # Termination
      done = (
          dist > 0.5
          or cube_pos[2] < 0.01
          or cube_pos[2] > 1.0
          or self.current_step >= self.max_steps
          )

      return obs, reward, done, False, {}





    def _compute_reward(self):
      cube_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")
      cube_pos = self.data.xpos[cube_id]
      palm_pos = self.data.body("right_hand_index_1_link").xpos

      dist = np.linalg.norm(palm_pos - cube_pos)
      cube_vel = np.linalg.norm(self.data.cvel[cube_id])

      # Count how many fingers are touching the cube
      fingers = [
        "right_hand_thumb_2_link",
        "right_hand_index_1_link",
        "right_hand_middle_1_link"
      ]
      touch_count = sum(self._is_touching(f, "cube") for f in fingers)

      # Grasp quality heuristic
      if touch_count == 0:
          grasp_quality = -1.0
      elif touch_count == 1:
          grasp_quality = 0.1
      elif touch_count == 2:
          grasp_quality = 0.4
      else:  # 3+
          grasp_quality = 0.9 if cube_vel < 0.05 else 0.5

      # Reward components
      reward = 0
      reward += 5.0 / (1.0 + 20 * dist)
      reward += 2.0 if dist < 0.06 else 0
      reward += 10.0 * grasp_quality
      reward -= 2.0 * min(1.0, cube_vel)
      reward -= 0.005  # time penalty

      # Debug
      print(f"[step {self.current_step}] dist: {dist:.3f}, vel: {cube_vel:.3f}, touches: {touch_count}, grasp_quality: {grasp_quality:.2f}, reward: {reward:.2f}")

      return reward





    def _get_obs(self):
          cube_pos = self.data.body("cube").xpos.copy()
          palm_pos = self.data.body("right_hand_index_1_link").xpos.copy()
          relative_pos = cube_pos - palm_pos
          base_state = np.concatenate([self.data.qpos, self.data.qvel])
          obs = np.concatenate([base_state, cube_pos, palm_pos, relative_pos])
          expected_dim = self.observation_space.shape[0]
          fixed_obs = np.zeros(expected_dim, dtype=np.float32)
          obs = obs.astype(np.float32)

          fixed_obs[:min(expected_dim, obs.shape[0])] = obs[:min(expected_dim, obs.shape[0])]
          return fixed_obs

    def _is_touching(self, geom1, geom2):
      for i in range(self.data.ncon):
        contact = self.data.contact[i]
        name1 = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_GEOM, contact.geom1)
        name2 = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_GEOM, contact.geom2)
        if (geom1 in (name1, name2)) and (geom2 in (name1, name2)):
            return True
      return False
    def _fingers_touching_cube(self):
      fingers = ["right_hand_thumb_2_link", "right_hand_index_1_link", "right_hand_middle_1_link"]
      touched = 0
      for f in fingers:
          if self._is_touching(f, "cube"):
              touched += 1
      return touched




In [29]:
env = CustomRobotEnv()
print("Actuator count:", env.model.nu)
print("Right actuator IDs:", env.right_actuator_ids)


Actuator count: 60
Right actuator IDs: [ 7  8  9 10 11 12 13]


In [28]:
import mujoco
from mujoco import MjModel

# Load your model
model = MjModel.from_xml_path("/home/oussema/Documents/project/results/g1_combined_ultra_stable.xml")

print("=== Actuator names in the model ===")
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"{i}: {name}")


=== Actuator names in the model ===
0: left_shoulder_pitch_joint
1: left_shoulder_roll_joint
2: left_shoulder_yaw_joint
3: left_elbow_joint
4: left_wrist_roll_joint
5: left_wrist_pitch_joint
6: left_wrist_yaw_joint
7: right_shoulder_pitch_joint
8: right_shoulder_roll_joint
9: right_shoulder_yaw_joint
10: right_elbow_joint
11: right_wrist_roll_joint
12: right_wrist_pitch_joint
13: right_wrist_yaw_joint
14: act_left_thumb_0
15: act_left_thumb_1
16: act_left_index_0
17: act_left_index_1
18: act_left_middle_0
19: act_left_middle_1
20: act_left_ring_0
21: act_left_ring_1
22: act_right_thumb_0
23: act_right_thumb_1
24: act_right_index_0
25: act_right_index_1
26: act_right_middle_0
27: act_right_middle_1
28: act_right_ring_0
29: act_right_ring_1
30: act_left_shoulder_pitch_joint
31: act_left_shoulder_roll_joint
32: act_left_shoulder_yaw_joint
33: act_left_elbow_joint
34: act_left_wrist_roll_joint
35: act_left_wrist_pitch_joint
36: act_left_wrist_yaw_joint
37: act_right_shoulder_pitch_joint
38

In [30]:
#  8. Define a Simple Training Log Callback
class PrintCallback(BaseCallback):
    def __init__(self, verbose=1):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        if self.n_calls % 100 == 0:
            print(f"Step {self.n_calls}, reward: {self.locals['rewards'][-1]:.3f}")
        return True

In [31]:
#  1. Import the noise module
from stable_baselines3.common.noise import NormalActionNoise

#  2. Create the environment first
env = CustomRobotEnv()

#  3. Now define the noise after env is available
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.3 * np.ones(n_actions))

#  4. Create the SAC model with noise
model_sac = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [32]:
import os
import imageio
from stable_baselines3.common.callbacks import BaseCallback
from datetime import datetime
from PIL import Image

class EvalVideoCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=50000, video_length=300, video_folder="videos/", prefix="grasp_eval", verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.video_length = video_length
        self.video_folder = video_folder
        self.prefix = prefix
        os.makedirs(video_folder, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            obs, _ = self.eval_env.reset()
            print(f"[DEBUG] obs.shape: {obs.shape}")  # ← ajoute ceci

            action, _ = self.model.predict(obs, deterministic=True)

            obs, _ = self.eval_env.reset()
            frames = []

            for _ in range(self.video_length):
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, done, _, _ = self.eval_env.step(action)

                self.eval_env.renderer.update_scene(self.eval_env.data)
                frame = self.eval_env.renderer.render()
                frames.append(Image.fromarray(frame.astype(np.uint8)))

                if done:
                    break

            # Save video
            timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
            video_path = os.path.join(
                self.video_folder, f"{self.prefix}_{self.n_calls}_steps_{timestamp}.mp4"
            )
            imageio.mimsave(video_path, frames, fps=30)
            print(f"🎥 Saved evaluation video: {video_path}")

        return True


In [38]:
env = CustomRobotEnv()
eval_env = CustomRobotEnv()

callback = EvalVideoCallback(
    eval_env=eval_env,
    eval_freq=50000,
    video_length=300,
    video_folder="/home/oussema/Documents/project/videos/",
    prefix="grasp_eval"
)

model_sac = TD3(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    batch_size=256,
    buffer_size=1_000_000,
    gamma=0.98,
    tau=0.02
)
obs, info = env.reset()
print("[TEST] obs.shape:", obs.shape)
print("[TEST] obs dtype:", type(obs), obs.dtype if isinstance(obs, np.ndarray) else None)

model_sac.learn(total_timesteps=50_000, callback=callback)


# Save model
model_sac.save("/home/oussema/Documents/project/sac_custom_robot")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


KeyError: "Invalid name 'right_hand_index_1_link'. Valid names: ['cube', 'left_elbow_link', 'left_index_0', 'left_index_1', 'left_index_force_sensors', 'left_middle_0', 'left_middle_1', 'left_middle_force_sensors', 'left_ring_0', 'left_ring_1', 'left_ring_force_sensors', 'left_shoulder_pitch_link', 'left_shoulder_roll_link', 'left_shoulder_yaw_link', 'left_thumb_0', 'left_thumb_1', 'left_thumb_force_sensors', 'left_wrist_pitch_link', 'left_wrist_roll_link', 'left_wrist_yaw_link', 'right_elbow_link', 'right_index_0', 'right_index_1', 'right_index_force_sensors', 'right_middle_0', 'right_middle_1', 'right_middle_force_sensors', 'right_ring_0', 'right_ring_1', 'right_ring_force_sensors', 'right_shoulder_pitch_link', 'right_shoulder_roll_link', 'right_shoulder_yaw_link', 'right_thumb_0', 'right_thumb_1', 'right_thumb_force_sensors', 'right_wrist_pitch_link', 'right_wrist_roll_link', 'right_wrist_yaw_link', 'table', 'world']"

In [17]:
# 10. Evaluate and Record Video (longer)
frames = []
obs, _ = env.reset()
for t in range(1000):  # increase number of steps/frames
    action, _ = model_sac.predict(obs, deterministic=True)
    obs, _, _, _, _ = env.step(action)
    env.renderer.update_scene(env.data)
    frame = env.renderer.render()
    frames.append(Image.fromarray(frame.astype(np.uint8)))

# Save at 30 fps
imageio.mimsave("custom_robot_eval.mp4", frames, fps=30)

ValueError: could not broadcast input array from shape (3,) into shape (1,)

In [21]:
# Download the video
video_path = "custom_robot_eval.mp4"
files.download(video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>